# Mutual Fund Performance and Risk Analytics

This notebook calculates key performance and risk metrics (CAGR, Volatility, Sharpe, Sortino, Alpha, Beta, Maximum Drawdown) for the 6 Large Cap mutual fund schemes in our database, ranks them, and evaluates them against the category average benchmark.

### Structure of the analysis:
1. Load NAV history and calculate Daily Returns
2. Calculate Compound Annual Growth Rate (CAGR) for 1-Year, 3-Year, and 5-Year horizons
3. Calculate risk metrics: Volatility and Maximum Drawdown (MDD)
4. Calculate risk-adjusted return ratios: Sharpe and Sortino Ratios
5. Calculate CAPM metrics: Alpha and Beta (relative to Category Benchmark)
6. Rank funds by risk and return metrics
7. Compute a Composite Fund Score
8. Identify Top and Bottom performing funds
9. Compare funds against the Category Benchmark
10. Export results to CSVs (`fund_scorecard.csv`, `alpha_beta.csv`, `risk_metrics.csv`)
11. Export comparison visualizations to the `reports/` folder

## 1. Load NAV History and Calculate Daily Returns
We load daily NAV history from the SQLite database `bluestock_mf.db` and calculate daily returns.

In [ ]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Locate database path
db_path = '../bluestock_mf.db' if os.path.exists('../bluestock_mf.db') else 'bluestock_mf.db'
conn = sqlite3.connect(db_path)
df_fund = pd.read_sql_query("SELECT * FROM dim_fund", conn)
df_nav = pd.read_sql_query("SELECT * FROM fact_nav", conn)
conn.close()

# Merge facts with fund details and pivot
df_nav_full = pd.merge(df_nav, df_fund, on='scheme_code', how='inner')
df_nav_full['date'] = pd.to_datetime(df_nav_full['date'])
df_nav_pivot = df_nav_full.pivot(index='date', columns='scheme_name', values='nav')
df_nav_pivot.columns = [col.split(' - ')[0] for col in df_nav_pivot.columns]

# Calculate daily returns
df_returns = df_nav_pivot.pct_change().dropna()

# Define Category Benchmark daily returns as the equal-weighted average of the 6 funds
df_returns['Category Benchmark'] = df_returns.mean(axis=1)
print("Daily returns calculated for all schemes and the Category Benchmark.")

## 2. Calculate CAGR (1-Year, 3-Year, and 5-Year)
Compound Annual Growth Rate (CAGR) measures the geometric returns over a given period.

In [ ]:
latest_date = df_nav_pivot.index.max()
print(f"Latest NAV Date in dataset: {latest_date.strftime('%Y-%m-%d')}")

cagr_list = []
for col in df_nav_pivot.columns:
    nav_series = df_nav_pivot[col].dropna()
    latest_nav = nav_series.loc[latest_date]
    
    # 1-Year lookback
    target_1y = latest_date - pd.Timedelta(days=365)
    idx_1y = np.abs(nav_series.index - target_1y).argmin()
    nav_1y = nav_series.iloc[idx_1y]
    cagr_1y = (latest_nav / nav_1y) - 1
    
    # 3-Year lookback
    target_3y = latest_date - pd.Timedelta(days=3 * 365)
    idx_3y = np.abs(nav_series.index - target_3y).argmin()
    nav_3y = nav_series.iloc[idx_3y]
    cagr_3y = (latest_nav / nav_3y) ** (1/3) - 1
    
    # 5-Year lookback
    target_5y = latest_date - pd.Timedelta(days=5 * 365)
    idx_5y = np.abs(nav_series.index - target_5y).argmin()
    nav_5y = nav_series.iloc[idx_5y]
    cagr_5y = (latest_nav / nav_5y) ** (1/5) - 1
    
    cagr_list.append({
        'scheme_name': col,
        'cagr_1y': round(cagr_1y * 100, 2),
        'cagr_3y': round(cagr_3y * 100, 2),
        'cagr_5y': round(cagr_5y * 100, 2)
    })

df_cagr = pd.DataFrame(cagr_list)
print(df_cagr)

## 3. Calculate Volatility and Maximum Drawdown
- Volatility is calculated as the annualized standard deviation of daily returns.
- Maximum Drawdown represents the largest peak-to-trough drop in NAV.

In [ ]:
risk_list = []
for col in df_nav_pivot.columns:
    # Annualized Volatility
    daily_vol = df_returns[col].std()
    ann_vol = daily_vol * np.sqrt(252)
    
    # Maximum Drawdown
    nav_series = df_nav_pivot[col].dropna()
    cum_max = nav_series.cummax()
    drawdown = (nav_series - cum_max) / cum_max
    max_dd = drawdown.min()
    
    risk_list.append({
        'scheme_name': col,
        'volatility': round(ann_vol * 100, 2),  # in %
        'max_drawdown': round(max_dd * 100, 2)  # in %
    })

df_risk = pd.DataFrame(risk_list)
print(df_risk)

## 4. Calculate Risk-Adjusted Returns (Sharpe and Sortino Ratios)
We assume a standard Risk-Free Rate of 6% (0.06) for risk-adjusted calculations.
- Sharpe Ratio rewards return relative to total volatility.
- Sortino Ratio rewards return relative only to downside volatility (negative returns).

In [ ]:
rf_rate = 0.06
ratio_list = []

for col in df_nav_pivot.columns:
    ret_series = df_returns[col]
    ann_return = ret_series.mean() * 252
    ann_vol = ret_series.std() * np.sqrt(252)
    
    # Sharpe Ratio
    sharpe = (ann_return - rf_rate) / ann_vol
    
    # Downside Volatility
    downside_rets = ret_series[ret_series < 0]
    downside_vol = downside_rets.std() * np.sqrt(252)
    
    # Sortino Ratio
    sortino = (ann_return - rf_rate) / downside_vol if downside_vol > 0 else np.nan
    
    ratio_list.append({
        'scheme_name': col,
        'annualized_return': round(ann_return * 100, 2),
        'sharpe_ratio': round(sharpe, 3),
        'sortino_ratio': round(sortino, 3)
    })

df_ratios = pd.DataFrame(ratio_list)
print(df_ratios)

## 5. Calculate CAPM Metrics (Alpha and Beta)
We calculate Alpha and Beta relative to the **Category Benchmark** (equal-weighted index of the 6 Large Cap funds).

In [ ]:
capm_list = []
bench_rets = df_returns['Category Benchmark']
ann_bench_ret = bench_rets.mean() * 252
var_bench = bench_rets.var()

for col in df_nav_pivot.columns:
    ret_series = df_returns[col]
    ann_return = ret_series.mean() * 252
    
    # Calculate Beta
    cov_bench = ret_series.cov(bench_rets)
    beta = cov_bench / var_bench
    
    # Calculate Alpha
    alpha = ann_return - (rf_rate + beta * (ann_bench_ret - rf_rate))
    
    capm_list.append({
        'scheme_name': col,
        'alpha': round(alpha * 100, 2),  # in %
        'beta': round(beta, 3)
    })

df_capm = pd.DataFrame(capm_list)
print(df_capm)

## 6. Risk and Return Rankings
We rank the schemes based on returns (3-Year CAGR) and risk (annualized volatility).

In [ ]:
# Merge results
df_all = pd.merge(df_cagr, df_risk, on='scheme_name')
df_all = pd.merge(df_all, df_ratios, on='scheme_name')
df_all = pd.merge(df_all, df_capm, on='scheme_name')

# Perform ranking
df_all['return_rank'] = df_all['cagr_3y'].rank(ascending=False).astype(int)
df_all['risk_rank'] = df_all['volatility'].rank(ascending=True).astype(int)  # lower volatility is better

print("Rankings completed:")
print(df_all[['scheme_name', 'cagr_3y', 'return_rank', 'volatility', 'risk_rank']])

## 7. Composite Fund Score
We define a Composite Fund Score based on the average percentile rank of: 3-Year CAGR, Sharpe Ratio, and Sortino Ratio. This provides a balanced assessment of return growth and risk adjusted return efficiency.

In [ ]:
# Calculate percentile ranks
cagr_pct = df_all['cagr_3y'].rank(pct=True)
sharpe_pct = df_all['sharpe_ratio'].rank(pct=True)
sortino_pct = df_all['sortino_ratio'].rank(pct=True)

# Score from 1 to 100
df_all['composite_score'] = round(((cagr_pct + sharpe_pct + sortino_pct) / 3) * 100, 1)

# Final Rank based on score
df_all['overall_rank'] = df_all['composite_score'].rank(ascending=False).astype(int)
print(df_all[['scheme_name', 'composite_score', 'overall_rank']].sort_values('overall_rank'))

## 8. Top and Bottom Performing Funds
We list the funds ranked from top to bottom based on their Composite Score.

In [ ]:
print("--- Top Funds (Sorted by Rank Ascending) ---")
print(df_all[['overall_rank', 'scheme_name', 'composite_score', 'cagr_3y', 'sharpe_ratio']].sort_values('overall_rank'))

print("\n--- Bottom Funds (Sorted by Rank Descending) ---")
print(df_all[['overall_rank', 'scheme_name', 'composite_score', 'cagr_3y', 'sharpe_ratio']].sort_values('overall_rank', ascending=False))

## 9. Benchmark Comparison
We compare our individual mutual funds against the Category Benchmark.

In [ ]:
# Reconstruct benchmark index starting at 100
bench_index = 100 * (1 + df_returns['Category Benchmark']).cumprod()
latest_bench_val = bench_index.loc[latest_date]

# Benchmark CAGR calculations
target_b_1y = latest_date - pd.Timedelta(days=365)
idx_b_1y = np.abs(bench_index.index - target_b_1y).argmin()
bench_cagr_1y = (latest_bench_val / bench_index.iloc[idx_b_1y]) - 1

target_b_3y = latest_date - pd.Timedelta(days=3 * 365)
idx_b_3y = np.abs(bench_index.index - target_b_3y).argmin()
bench_cagr_3y = (latest_bench_val / bench_index.iloc[idx_b_3y]) ** (1/3) - 1

target_b_5y = latest_date - pd.Timedelta(days=5 * 365)
idx_b_5y = np.abs(bench_index.index - target_b_5y).argmin()
bench_cagr_5y = (latest_bench_val / bench_index.iloc[idx_b_5y]) ** (1/5) - 1

# Benchmark Volatility & Max Drawdown
bench_vol = df_returns['Category Benchmark'].std() * np.sqrt(252)
bench_cum_max = bench_index.cummax()
bench_mdd = ((bench_index - bench_cum_max) / bench_cum_max).min()
bench_sharpe = (df_returns['Category Benchmark'].mean() * 252 - rf_rate) / bench_vol

# Append Benchmark row to compare side-by-side
bench_row = {
    'scheme_name': 'Category Benchmark Proxy',
    'cagr_1y': round(bench_cagr_1y * 100, 2),
    'cagr_3y': round(bench_cagr_3y * 100, 2),
    'cagr_5y': round(bench_cagr_5y * 100, 2),
    'volatility': round(bench_vol * 100, 2),
    'max_drawdown': round(bench_mdd * 100, 2),
    'sharpe_ratio': round(bench_sharpe, 3),
    'alpha': 0.0,
    'beta': 1.0,
    'composite_score': np.nan,
    'overall_rank': np.nan
}

df_comparison = pd.concat([df_all, pd.DataFrame([bench_row])], ignore_index=True)
print(df_comparison[['scheme_name', 'cagr_3y', 'volatility', 'sharpe_ratio', 'alpha', 'beta']])

## 10. Generate CSV Reports
We save three CSV reports (`fund_scorecard.csv`, `alpha_beta.csv`, `risk_metrics.csv`) to `data/processed/` directory.

In [ ]:
processed_dir = '../data/processed/' if os.path.exists('../data/processed/') else 'data/processed/'
os.makedirs(processed_dir, exist_ok=True)

# 1. fund_scorecard.csv
scorecard_cols = ['scheme_name', 'cagr_1y', 'cagr_3y', 'cagr_5y', 'composite_score', 'overall_rank']
df_all[scorecard_cols].to_csv(os.path.join(processed_dir, 'fund_scorecard.csv'), index=False)

# 2. alpha_beta.csv
alpha_beta_cols = ['scheme_name', 'alpha', 'beta']
df_all[alpha_beta_cols].to_csv(os.path.join(processed_dir, 'alpha_beta.csv'), index=False)

# 3. risk_metrics.csv
risk_cols = ['scheme_name', 'volatility', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown']
df_all[risk_cols].to_csv(os.path.join(processed_dir, 'risk_metrics.csv'), index=False)

print("All three CSV reports exported successfully to data/processed/.")

## 11. Export Comparison Charts
We save performance and risk comparison plots to the `reports/` directory.

In [ ]:
reports_dir = '../reports/' if os.path.exists('../reports/') else 'reports/'
os.makedirs(reports_dir, exist_ok=True)

# 1. Performance vs Risk Scatter Plot (3Y CAGR vs Volatility)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_all, x='volatility', y='cagr_3y', hue='scheme_name', s=150, palette='Set1')
plt.title('Performance vs Risk: 3-Year CAGR vs Annualized Volatility', fontsize=14)
plt.xlabel('Annualized Volatility (%)')
plt.ylabel('3-Year CAGR (%)')
plt.grid(True, linestyle='--')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'performance_vs_risk.png'))
plt.show()

# 2. Composite Score Comparison
df_all_sorted = df_all.sort_values('composite_score', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_all_sorted, x='composite_score', y='scheme_name', palette='crest')
plt.title('Composite Fund Score Comparison (Higher is Better)', fontsize=14)
plt.xlabel('Composite Score')
plt.ylabel('Mutual Fund Scheme')
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'composite_scores.png'))
plt.show()

print("Visualizations exported successfully to reports/.")